# Fine-tune DistilBERT on HC3 (human vs. ChatGPT)

**Run this on Colab with a GPU runtime** (Runtime -> Change runtime type -> T4 GPU).
Local CPU benchmarking showed ~1 sample/sec at seq_len=256, which makes the full
dataset (23k train examples x 3 epochs x 2 length variants) an 18+ hour job on CPU.
On a T4 this should take well under an hour per variant.

This notebook is self-contained: it re-downloads and re-processes HC3 directly
(no need to upload local files), trains two variants —

- **length_controlled**: human/chatgpt answer pairs truncated to equal word count
  (removes the length shortcut — see the note on word-count distributions below)
- **raw**: original, uncontrolled lengths (kept only to demonstrate via
  interpretability that an unconstrained model learns to exploit length)

— and saves both checkpoints to Google Drive so they can be pulled back down
for the interpretability and cross-model notebooks.

**Background**: HC3 pairs are topic-matched by construction (each question has
a human and a ChatGPT answer), but exploration showed ChatGPT answers are
systematically longer and lower-variance (median ~173 vs ~82 words on
reddit_eli5, std ~62 vs ~164) — a classifier could cheat on length alone
without learning anything about style. The length_controlled variant truncates
both sides of each pair to `min(len_human, len_chatgpt, max_words)` words.

In [ ]:
!pip install -q -U transformers datasets huggingface_hub accelerate pyarrow scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/ai_text_detection_checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)

print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> GPU before running training.'

## Data loading (mirrors `src/data_utils.py`)

Inlined here so this notebook runs standalone on Colab without needing the
repo files uploaded.

In [ ]:
import re

HC3_REPO_ID = 'Hello-SimpleAI/HC3'
HC3_FILENAME = 'all.jsonl'
DOMAINS = ('reddit_eli5',)


def load_hc3():
    path = hf_hub_download(repo_id=HC3_REPO_ID, filename=HC3_FILENAME, repo_type='dataset')
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


_ESCAPED_NEWLINE_RE = re.compile(r'\\r\\n|\\n|\\r')
_WHITESPACE_RE = re.compile(r'\s+')


def _clean_text(text):
    # 10.6% of raw chatgpt_answers contain a literal two-character "\n" (paragraph
    # break stored as text, never converted to whitespace) vs. 0.0% of human_answers.
    # That's a data-collection artifact, not style - interpretability analysis showed
    # it was the single strongest token pushing predictions toward "AI". Strip it
    # before any downstream processing so it can't be learned as a shortcut.
    text = _ESCAPED_NEWLINE_RE.sub(' ', text)
    return _WHITESPACE_RE.sub(' ', text).strip()


def _truncate_to_words(text, n_words):
    words = text.split()
    return ' '.join(words[:n_words])


_SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')


def _truncate_to_words_sentence_aware(text, n_words):
    # Truncate to the last complete sentence at/before n_words. A hard word-count
    # cutoff leaves a systematic tell (truncated side almost always ends mid-sentence),
    # which is itself a shortcut nearly as strong as raw length - so we always stop at
    # a sentence boundary, even if that means slightly overshooting n_words when the
    # first sentence alone is longer than the budget.
    sentences = [s for s in _SENTENCE_SPLIT_RE.split(text.strip()) if s]
    kept_words = []
    for sentence in sentences:
        sentence_words = sentence.split()
        if kept_words and len(kept_words) + len(sentence_words) > n_words:
            break
        kept_words.extend(sentence_words)
        if len(kept_words) >= n_words:
            break
    return ' '.join(kept_words)


def flatten_to_binary(df, domains=DOMAINS, length_control=True, max_words=200, min_words=5, seed=42):
    sub = df[df['source'].isin(domains)].reset_index(drop=True)
    records = []
    for pair_id, row in sub.iterrows():
        human_list = row['human_answers'] or []
        chatgpt_list = row['chatgpt_answers'] or []
        if not human_list or not chatgpt_list:
            continue
        human_text = _clean_text(human_list[0])
        chatgpt_text = _clean_text(chatgpt_list[0])
        if not human_text or not chatgpt_text:
            continue
        human_wc = len(human_text.split())
        chatgpt_wc = len(chatgpt_text.split())
        if human_wc < min_words or chatgpt_wc < min_words:
            continue
        if length_control:
            target_len = min(human_wc, chatgpt_wc, max_words)
            human_text = _truncate_to_words_sentence_aware(human_text, target_len)
            chatgpt_text = _truncate_to_words_sentence_aware(chatgpt_text, target_len)
            human_wc = len(human_text.split())
            chatgpt_wc = len(chatgpt_text.split())
        records.append(dict(pair_id=pair_id, source=row['source'], question=row['question'], text=human_text, label=0, word_count=human_wc))
        records.append(dict(pair_id=pair_id, source=row['source'], question=row['question'], text=chatgpt_text, label=1, word_count=chatgpt_wc))
    out = pd.DataFrame(records)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)


def group_train_val_test_split(df, group_col='pair_id', test_size=0.15, val_size=0.15, seed=42):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_val_idx, test_idx = next(gss1.split(df, groups=df[group_col]))
    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    relative_val_size = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(train_val_df, groups=train_val_df[group_col]))
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, test_df


raw_df = load_hc3()
print('raw shape:', raw_df.shape)

In [ ]:
class HC3Dataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(list(texts), truncation=True, padding='max_length', max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}


MODEL_NAME = 'distilbert-base-uncased'


def train_variant(length_control, tag, epochs=3, batch_size=16, lr=2e-5, max_length=256):
    flat = flatten_to_binary(raw_df, length_control=length_control)
    train_df, val_df, test_df = group_train_val_test_split(flat)
    print(f'[{tag}] total={flat.shape[0]} train={train_df.shape[0]} val={val_df.shape[0]} test={test_df.shape[0]}')

    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    train_ds = HC3Dataset(train_df['text'], train_df['label'], tokenizer, max_length)
    val_ds = HC3Dataset(val_df['text'], val_df['label'], tokenizer, max_length)
    test_ds = HC3Dataset(test_df['text'], test_df['label'], tokenizer, max_length)

    output_dir = f'{CHECKPOINT_ROOT}/{tag}'
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_steps=50,
        report_to='none',
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    val_metrics = trainer.evaluate()
    test_metrics = trainer.evaluate(test_ds)
    print(f'[{tag}] val:', val_metrics)
    print(f'[{tag}] test:', test_metrics)

    final_dir = f'{output_dir}/final'
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)

    # also persist the held-out test split so downstream notebooks (interpretability,
    # cross-model test) evaluate on exactly the same examples
    test_df.to_parquet(f'{final_dir}/test_split.parquet', index=False)
    print(f'[{tag}] saved to {final_dir}')
    return {'val_metrics': val_metrics, 'test_metrics': test_metrics, 'final_dir': final_dir}

## Train the main model (length-controlled)

This is the model used for the primary accuracy result and the cross-model
generalization test.

In [ ]:
results_length_controlled = train_variant(length_control=True, tag='length_controlled', epochs=3)

## Train the ablation model (raw, uncontrolled lengths)

Used only in the interpretability notebook to show what the model keys on
when the length shortcut is available.

In [ ]:
results_raw = train_variant(length_control=False, tag='raw', epochs=3)

In [ ]:
summary = pd.DataFrame([
    {'variant': 'length_controlled', **results_length_controlled['test_metrics']},
    {'variant': 'raw', **results_raw['test_metrics']},
])
print(summary)
summary.to_json(f'{CHECKPOINT_ROOT}/summary_metrics.json', orient='records', indent=2)
print(f"\nDone. Download the '{CHECKPOINT_ROOT}' folder from Drive (or sync it locally) \n"
      "into results/checkpoints/ before running 04_interpretability.ipynb and 05_cross_model_test.ipynb.")